# Reproducible scripts for Hydrogen Storage and Carriers

This companion notebook was generated from fenced `python` code blocks in `chapters/ch17_storage_and_carriers/chapter.md`. Blocks marked `noexec` in the chapter are preserved for reading but are not executed here.


In [1]:
from pathlib import Path
import re
import shutil

try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = None

_CHAPTER_FIGURES_DIR = Path("../figures")
_CHAPTER_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
_PAPERLAB_PATTERNS = ("*.png", "*.jpg", "*.jpeg", "*.svg", "*.webp", "*.csv", "*.json", "*.txt", "*.html")
_PAPERLAB_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}


def _paperlab_stamp(path):
    try:
        stat = path.stat()
        return (stat.st_mtime_ns, stat.st_size)
    except OSError:
        return None


def _paperlab_scan_files():
    files = {}
    roots = [Path("."), _CHAPTER_FIGURES_DIR]
    for root in roots:
        if not root.exists():
            continue
        for pattern in _PAPERLAB_PATTERNS:
            for path in root.glob(pattern):
                if path.is_file():
                    files[path.resolve()] = _paperlab_stamp(path)
    return files


def _paperlab_safe_name(name):
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("._")
    return cleaned or "notebook_output"


_paperlab_seen_files = _paperlab_scan_files()


def _paperlab_capture_new_files(label):
    global _paperlab_seen_files
    current = _paperlab_scan_files()
    changed = []
    for path, stamp in current.items():
        if _paperlab_seen_files.get(path) != stamp:
            changed.append(Path(path))
    _paperlab_seen_files = current

    for path in sorted(changed):
        suffix = path.suffix.lower()
        if suffix in _PAPERLAB_IMAGE_SUFFIXES:
            if path.parent.resolve() == _CHAPTER_FIGURES_DIR.resolve():
                dest = path
            else:
                dest = _CHAPTER_FIGURES_DIR / _paperlab_safe_name(f"{label}_{path.name}")
                shutil.copy2(path, dest)
            print(f"Captured figure: figures/{dest.name}")
            if Image is not None and display is not None:
                display(Image(filename=str(dest)))
        else:
            print(f"Generated result file: {path}")

## Example 1 from `chapters/ch17_storage_and_carriers/chapter.md` line 116


This block is preserved but not executed because it is noexec marker.

````python
from neqsim import jneqsim as J

# Direct Java access through neqsim-python. Use explicit units and call
# setMixingRule before running flashes or process equipment.
# Storage and carrier comparison: gravimetric density, volumetric density,
# and the round-trip energy penalty (charge plus discharge as a fraction of LHV).
options = []

# 1) Compressed gaseous H2 at 700 bar, 25 degC
cgh2 = J.thermo.system.SystemSrkEos(298.15, 700.0)
cgh2.addComponent("hydrogen", 1.0)
cgh2.setMixingRule("classic")
ops = J.thermodynamicoperations.ThermodynamicOperations(cgh2)
ops.TPflash()
cgh2.initProperties()
options.append({
    "option": "CGH2 700 bar",
    "rho_vol_kg_per_m3": cgh2.getDensity("kg/m3"),
    "gravimetric_wt_pct": 100.0,
    "round_trip_penalty_frac_LHV": 0.15,
})

# 2) Liquid H2 at 20.3 K, 1 bar (use Leachman EOS for normal hydrogen)
lh2 = J.thermo.system.SystemLeachmanEos(20.3, 1.0)
lh2.addComponent("hydrogen", 1.0)
lh2.init(0)
options.append({
    "option": "LH2 20K",
    "rho_vol_kg_per_m3": lh2.getPhase(0).getDensity_Leachman("normal"),
    "gravimetric_wt_pct": 100.0,
    "round_trip_penalty_frac_LHV": 0.35,
})

# 3) Ammonia carrier (NH3 -> N2 + 3/2 H2, 17.6 wt% H2)
nh3 = J.thermo.system.SystemSrkEos(298.15, 10.0)
nh3.addComponent("ammonia", 1.0)
nh3.setMixingRule("classic")
ops = J.thermodynamicoperations.ThermodynamicOperations(nh3)
ops.TPflash()
nh3.initProperties()
options.append({
    "option": "Ammonia carrier",
    "rho_vol_kg_per_m3": nh3.getDensity("kg/m3") * 0.176,  # H2 equivalent volumetric
    "gravimetric_wt_pct": 17.6,
    "round_trip_penalty_frac_LHV": 0.50,
})

for opt in options:
    print(opt)

````
